# Musanga V4 — NB00: RGB / DSM / SfM inventory and input audit

**Inherited from V3_1 NB00:** input auditing and source-data provenance. V3_1 has already approved the KVO field-grounded annotations; **do not reapprove, refit or replace them here.**

**Primary V4 hypothesis:** RGB + high-resolution DSM-derived structure improves held-out Musanga segmentation over RGB-only. LAS-derived 3D descriptors are a low-cost exploratory ablation only. No DTM, CHM, or under-canopy layer is assumed from SfM.

**Output contract:** `Final/Outputs/v4/00_input_audit/`. All earlier output directories and source rasters are read-only.

**Reproduction requirements:** same V3_1 30-circle fold membership (excluded original IDs 6, 9, 24, 25); preserve the 15 m spatial training exclusion, annotation semantics, and MOS 3–4 exclusion in the NB02B benchmark. This notebook **does not regenerate folds, train models, merge LAS, or process full point clouds**.

**Go/no-go checks:** confirm each RGB mosaic's intended DSM partner, exact CRS and coordinate transform, DSM valid coverage, SfM overlap, and acquisition chronology before enabling any V4 model-training notebook.

## 00 — Environment and read-only source registries

In [ ]:
%pip -q install rasterio geopandas pyogrio shapely pandas numpy matplotlib laspy
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import json, hashlib, re, warnings
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
from IPython.display import display
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass

ROOT_CANDIDATES=[
    Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
    Path('/content/drive/MyDrive/Musanga/Final'),
]
ROOT=next((p for p in ROOT_CANDIDATES if p.exists()),None)
assert ROOT is not None, f'Could not find Final in {ROOT_CANDIDATES}'
DATA=ROOT/'Data'
assert DATA.exists(),f'Data is missing: {DATA}'
V31=ROOT/'Outputs/V3_1'
V31_MANIFEST=V31/'00_input_audit/00_input_manifest.json'
assert V31_MANIFEST.exists(),f'Approved V3_1 input manifest missing: {V31_MANIFEST}'
base=json.loads(V31_MANIFEST.read_text())
assert base.get('schema_version')=='v3_1_input_audit_kvo_final', base.get('schema_version')
OUT=ROOT/'Outputs/v4/00_input_audit'
OUT.mkdir(parents=True,exist_ok=True)
print('DATA (read-only):',DATA)
print('V3_1 manifest (read-only):',V31_MANIFEST)
print('V4 output:',OUT)
print('V3_1 annotation reference:',base.get('reference_version'))


## 01 — Discover RGB mosaics, DSM rasters and LAS without guessing site mappings

In [ ]:
FORESTS=('Baego','MOS','Yangambi','Yoko')
# Candidates only. Nothing is approved from filename matching alone.
SITE_KEYWORDS={
    'Baego': ('baego',),
    'MOS': ('mos_', 'mos-', 'mos1', 'mos5', 'mos 1', 'mos 5', 'mlt_mos'),
    'Yangambi': ('yangambi','carlier2020','ybjoseph','ybforgeo'),
    'Yoko': ('yoko',),
}
RGB_NAMES=[
    'Baego.tif',
    'Mos_1_2_transparent_mosaic_group1_georef.tif',
    'Mos_5_6_transparent_mosaic_group1_georef.tif',
    'Yangambi_georef.tif',
    'Yoko_kvo_georef.tif',
]
LAS_EXPECTED={
    'MOS': ['mlt_Mos_1_2_Green_densified_point_cloud_georef.las',
            'mlt_Mos_3_4_Green_densified_point_cloud_georef.las',
            'mlt_Mos_5_6_Green_densified_point_cloud_georef.las'],
    'Yangambi': ['Carlier2020_Aug20_flights_group1_densified_point_cloud_part_'+str(i)+'_georef.las' for i in range(1,7)]
              + ['YBJoseph_Green_densified_point_cloud_georef.las',
                 'YBForgeo_Green_densified_point_cloud_georef.las'],
    'Yoko': ['Yoko_survey_georef.las'],
    'Baego': [
        'Nadine_Baego_B2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_B3mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C2mlt_2023_Green_densified_point_cloud.las',
        'Nadine_Baego_C3mlt_2023_Green_densified_point_cloud.las',
    ],
}
# Bounded to project Data; metadata-only scan, no heavy raster/LAS loads.
paths=[p for p in DATA.rglob('*') if p.is_file() and p.suffix.lower() in ('.tif','.tiff','.las','.laz')]
rows=[]
for p in paths:
    name=p.name.lower()
    tags=[site for site, keys in SITE_KEYWORDS.items() if any(k in name or k in str(p.parent).lower() for k in keys)]
    rows.append({'path':str(p),'name':p.name,'type':'LAS' if p.suffix.lower() in ('.las','.laz') else 'raster',
                 'site_candidate':'|'.join(tags) if tags else '',
                 'dsm_candidate':bool(re.search(r'(^|[_\\-])(dsm|surface|elevation)([_\\-]|$)',p.stem,re.I)),
                 'bytes':p.stat().st_size})
inventory=pd.DataFrame(rows,columns=['path','name','type','site_candidate','dsm_candidate','bytes'])
inventory.to_csv(OUT/'00_file_inventory.csv',index=False)
display(inventory.groupby(['type','site_candidate'],dropna=False).size().rename('n').reset_index())
print('Rasters with DSM-style names:')
display(inventory.query('dsm_candidate').loc[:,['site_candidate','name','path']])
print('Known V3_1 active RGB mosaics:')
for name in RGB_NAMES:
    matches=inventory[inventory.name.str.lower()==name.lower()]
    print(name,'=>',matches.path.tolist())


## 02 — Inspect raster metadata and sample the valid-data fraction (no full-resolution reads)

In [ ]:
def raster_info(path):
    with rasterio.open(path) as ds:
        h,w=ds.height,ds.width
        # At most 512 x 512 per band; avoid full raster reads on Drive.
        sample=ds.read(1,out_shape=(min(512,h),min(512,w)),masked=True)
        vals=np.asarray(sample.compressed(),dtype=np.float64)
        b=ds.bounds
        return {'path':str(path),'name':Path(path).name,'crs':str(ds.crs) if ds.crs else None,
                'width':w,'height':h,'bands':ds.count,'px_x':ds.transform.a,'px_y':ds.transform.e,
                'left':b.left,'bottom':b.bottom,'right':b.right,'top':b.top,
                'nodata':ds.nodata,'sample_valid_fraction':float(np.ma.count(sample)/sample.size),
                'sample_z_min':float(np.nanmin(vals)) if len(vals) else np.nan,
                'sample_z_p50':float(np.nanmedian(vals)) if len(vals) else np.nan,
                'sample_z_p99':float(np.nanpercentile(vals,99)) if len(vals) else np.nan}
raster_rows=[]
for p in inventory.query("type == 'raster'").path:
    try:
        raster_rows.append({**raster_info(p),'error':''})
    except Exception as exc:
        raster_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
raster_df=pd.DataFrame(raster_rows)
raster_df.to_csv(OUT/'00_raster_metadata.csv',index=False)
display(raster_df[['name','crs','px_x','px_y','bands','sample_valid_fraction','error']].head(80))


## 03 — Explicit site-to-DSM selection and spatial overlap audit

In [ ]:
# IMPORTANT: edit ONLY this mapping after inspecting metadata. Do not accept automatic fuzzy matches.
# Use absolute paths or paths relative to Final/Data. Leave blank to mark unconfirmed.
DSM_BY_RGB={
    'Baego.tif': '',
    'Mos_1_2_transparent_mosaic_group1_georef.tif': '',
    'Mos_5_6_transparent_mosaic_group1_georef.tif': '',
    'Yangambi_georef.tif': '',
    'Yoko_kvo_georef.tif': '',
}
def resolve_data_path(s):
    if not s:return None
    p=Path(s)
    return p if p.is_absolute() else DATA/p
def intersection_fraction(a,b):
    xl=max(a[0],b[0]); yb=max(a[1],b[1]); xr=min(a[2],b[2]); yt=min(a[3],b[3])
    if xr<=xl or yt<=yb:return 0.
    return (xr-xl)*(yt-yb)/max((a[2]-a[0])*(a[3]-a[1]),1e-9)
overlap=[]
for name in RGB_NAMES:
    matches=inventory[inventory.name.str.lower()==name.lower()]
    dst=resolve_data_path(DSM_BY_RGB[name])
    rec={'rgb':name,'rgb_matches':len(matches),'dsm':str(dst) if dst else '',
         'approved_pair':False,'overlap_fraction_of_rgb_bounds':np.nan,'reason':''}
    if len(matches)!=1:
        rec['reason']='RGB missing or ambiguous'
    elif dst is None:
        rec['reason']='DSM selection pending'
    elif not dst.is_file():
        rec['reason']='DSM path does not exist'
    else:
        with rasterio.open(matches.iloc[0].path) as rgb, rasterio.open(dst) as dsm:
            if not rgb.crs or not dsm.crs:
                rec['reason']='missing CRS'
            else:
                dsm_bounds=transform_bounds(dsm.crs,rgb.crs,*dsm.bounds,densify_pts=21)
                rec['overlap_fraction_of_rgb_bounds']=intersection_fraction(rgb.bounds,dsm_bounds)
                rec['approved_pair']=bool(rec['overlap_fraction_of_rgb_bounds']>=0.95)
                rec['reason']='spatial extent overlap checked; subpixel registration NOT yet proven'
    overlap.append(rec)
pair_df=pd.DataFrame(overlap)
pair_df.to_csv(OUT/'00_rgb_dsm_pair_audit.csv',index=False)
display(pair_df)
assert all(not s for s in DSM_BY_RGB.values()) or pair_df.approved_pair.all(), (
    'One or more selected RGB–DSM pairs are unresolved; inspect pair audit.')


## 04 — SfM LAS header-only audit, including Yangambi's six Carlier parts

In [ ]:
import laspy
las_rows=[]
for p in inventory.query("type == 'LAS'").path:
    try:
        with laspy.open(p) as lf:
            h=lf.header
            las_rows.append({'path':p,'name':Path(p).name,'point_count':int(h.point_count),
                             'crs':str(h.parse_crs()) if h.parse_crs() else '',
                             'xmin':float(h.mins[0]),'ymin':float(h.mins[1]),'zmin':float(h.mins[2]),
                             'xmax':float(h.maxs[0]),'ymax':float(h.maxs[1]),'zmax':float(h.maxs[2]),
                             'point_format':str(h.point_format.id),'version':str(h.version),'error':''})
    except Exception as exc:
        las_rows.append({'path':p,'name':Path(p).name,'error':repr(exc)})
las_df=pd.DataFrame(las_rows)
las_df.to_csv(OUT/'00_las_header_inventory.csv',index=False)
display(las_df)
expected=[(site,name) for site,names in LAS_EXPECTED.items() for name in names]
missing=[(site,name) for site,name in expected if name.lower() not in set(inventory.name.str.lower())]
print('Expected LAS missing from local Data/LAS:',missing)
print('Interpretation: SfM surface reconstruction, not penetrating lidar; do NOT infer DTM, CHM, or foliage profiles.')
print('Carlier parts 1–6 are one logical Yangambi dataset. Do not physically merge until CRS, coordinates and overlap pass.')


## 05 — Freeze proposed V4 audit contract (no modelling yet)

In [ ]:
audit={
    'schema_version':'v4_nb00_rgb_dsm_sfm_inventory_1',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'inherited_reference_manifest':str(V31_MANIFEST),
    'inherited_reference_schema':base.get('schema_version'),
    'inherited_reference_version':base.get('reference_version'),
    'v3_1_folds_policy':'Reuse frozen 30-circle within-site spatial folds; do not regenerate',
    'v3_1_excluded_original_circle_ids':[6,9,24,25],
    'v3_1_training_exclusion_buffer_m':15,
    'primary_modalities':['RGB','DSM-derived local geometry'],
    'experimental_las':'small stratified crown sample, only if added information beyond DSM',
    'prohibited_unjustified_features':['CHM','DTM','understory profile from SfM'],
    'output_root':str(ROOT/'Outputs/v4'),
    'n_indexed_rasters':int((inventory.type=='raster').sum()),
    'n_indexed_las':int((inventory.type=='LAS').sum()),
    'rgb_dsm_pairings_confirmed':bool(len(pair_df)==len(RGB_NAMES) and pair_df.approved_pair.all()),
    'next_step':'N01 after RGB–DSM identity, coverage, registration and dates pass QA',
}
manifest=OUT/'00_v4_input_audit_manifest.json'
manifest.write_text(json.dumps(audit,indent=2))
print('WROTE:',manifest)
print('RGB–DSM pairing status:',audit['rgb_dsm_pairings_confirmed'])
print('V3_1 remains unchanged:',V31_MANIFEST)
if not audit['rgb_dsm_pairings_confirmed']:
    print('ACTION REQUIRED: fill DSM_BY_RGB after reviewing 00_raster_metadata.csv; then rerun Sections 03 and 05.')
